# 🩺 DiabèteCheck — Analyse & Entraînement du Modèle ML

**Projet Simplon Développeur Data IA**

Ce notebook couvre les 5 étapes du **Bloc 2 — Entraînement de modèle** :
1. Définition du problème ML
2. Préparation des données
3. Entraînement de 3 modèles
4. Évaluation avec métriques
5. Amélioration (GridSearchCV) et export

## 1. 📦 Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import joblib, json, os, urllib.request, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Bibliothèques chargées')

## 2. 📊 Chargement des données

**Source** : Pima Indians Diabetes Database (UCI / Kaggle)
- 768 patientes (femmes d'origine Pima, Arizona)
- 8 variables médicales
- Variable cible : `Outcome` (0 = pas de diabète, 1 = diabète)

In [ ]:
COLUMNS = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
DATA_PATH = '../data/diabetes.csv'
DATA_URL  = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'

os.makedirs('../data', exist_ok=True)
if not os.path.exists(DATA_PATH):
    print('⬇️  Téléchargement du dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print('✅ Dataset téléchargé')

df = pd.read_csv(DATA_PATH, names=COLUMNS)
print(f'📊 Dataset : {df.shape[0]} patients, {df.shape[1]-1} variables')
df.head()

## 3. 🔍 Analyse Exploratoire (EDA)

### 3.1 Informations générales

In [ ]:
print('=== TYPES ET VALEURS NULLES ===')
df.info()
print('\n=== STATISTIQUES DESCRIPTIVES ===')
df.describe().round(2)

### 3.2 Répartition des classes

In [ ]:
counts = df['Outcome'].value_counts()
print('Répartition des classes :')
print(f'  Non diabétique (0) : {counts[0]} ({counts[0]/len(df)*100:.1f}%)')
print(f'  Diabétique     (1) : {counts[1]} ({counts[1]/len(df)*100:.1f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.pie(counts, labels=['Non diabétique', 'Diabétique'],
        autopct='%1.1f%%', colors=['#1976D2', '#C62828'], startangle=90)
ax1.set_title('Répartition des classes')

# Convertir Outcome en string pour le palette seaborn
df_plot = df.copy()
df_plot['Outcome_str'] = df_plot['Outcome'].map({0: 'Non DT', 1: 'Diabétique'})
sns.boxplot(data=df_plot, x='Outcome_str', y='Glucose', ax=ax2,
            order=['Non DT', 'Diabétique'],
            palette={'Non DT': '#1976D2', 'Diabétique': '#C62828'})
ax2.set_xlabel('')
ax2.set_title('Glucose selon le diagnostic')
plt.tight_layout()
plt.savefig('../data/class_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.3 Valeurs manquantes (zéros impossibles médicalement)

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('⚠️  Valeurs nulles encodées comme zéros :')
for col in zero_cols:
    n = (df[col] == 0).sum()
    print(f'  {col:30s}: {n:3d} zéros ({n/len(df)*100:.1f}%)')

### 3.4 Distribution des variables

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Distribution des Variables par Diagnostic', fontsize=14, y=1.01)
colors = {0: '#1976D2', 1: '#C62828'}

for i, col in enumerate(COLUMNS[:-1]):
    ax = axes[i // 3][i % 3]
    for outcome in [0, 1]:
        df[df['Outcome'] == outcome][col].hist(
            bins=20, ax=ax, alpha=0.6,
            label='Non DT' if outcome == 0 else 'Diabétique',
            color=colors[outcome]
        )
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/distributions.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.5 Matrice de corrélation

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': .8})
plt.title('Matrice de Corrélation', fontsize=13)
plt.tight_layout()
plt.savefig('../data/correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n🔎 Variables les plus corrélées avec Outcome :')
print(corr['Outcome'].sort_values(ascending=False).drop('Outcome'))

## 4. 🔧 Préparation des données

In [ ]:
# Remplacer les zéros par la médiane par classe (imputation médiane stratifiée)
df_clean = df.copy()
# Convertir en float pour accepter les médianes décimales
for col in zero_cols:
    df_clean[col] = df_clean[col].astype(float)

for col in zero_cols:
    for outcome in [0, 1]:
        median = df[df['Outcome'] == outcome][col].replace(0, np.nan).median()
        mask_zero = (df['Outcome'] == outcome) & (df[col] == 0)
        df_clean.loc[mask_zero, col] = median

print('✅ Zéros remplacés par la médiane par classe')

# Vérification
for col in zero_cols:
    n = (df_clean[col] == 0).sum()
    print(f'  {col}: {n} zéros restants')

In [ ]:
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

# Division train/test 80/20 avec stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation (StandardScaler)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train : {len(X_train)} | Test : {len(X_test)}')
print(f'✅ Distribution train — Non DT: {(y_train==0).sum()} | DT: {(y_train==1).sum()}')
print(f'✅ Distribution test  — Non DT: {(y_test==0).sum()}  | DT: {(y_test==1).sum()}')

## 5. 🤖 Entraînement et comparaison de 3 modèles

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM':                 SVC(random_state=42, probability=True)
}

results = {}
print(f'{"Modèle":30s} | Accuracy | Précision | Rappel  | F1     | ROC-AUC')
print('-' * 80)
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'model':     model
    }
    r = results[name]
    print(f'{name:30s} | {r["accuracy"]:.3f}   | {r["precision"]:.3f}     | {r["recall"]:.3f}   | {r["f1"]:.3f}  | {r["roc_auc"]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names  = list(results.keys())
colors = ['#1976D2', '#2E7D32', '#E65100']

for ax, metric, title in [
    (axes[0], 'accuracy', 'Accuracy'),
    (axes[1], 'roc_auc',  'ROC-AUC')
]:
    vals = [results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(f'{title} par modèle', fontsize=11)
    ax.set_ylim(0.6, 1.0)
    ax.bar_label(bars, labels=[f'{v:.3f}' for v in vals], padding=3)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('Comparaison des Modèles', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. 📈 Évaluation détaillée du meilleur modèle

In [ ]:
best_name  = max(results, key=lambda k: results[k]['roc_auc'])
best_model = results[best_name]['model']
print(f'🏆 Meilleur modèle : {best_name} (ROC-AUC = {results[best_name]["roc_auc"]:.3f})')
print()
print('=== RAPPORT DE CLASSIFICATION ===')
y_pred_best  = best_model.predict(X_test_sc)
y_proba_best = best_model.predict_proba(X_test_sc)[:, 1]
print(classification_report(y_test, y_pred_best, target_names=['Non diabétique', 'Diabétique']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Non DT', 'DT'], yticklabels=['Non DT', 'DT'],
            linewidths=0.5)
ax1.set_title(f'Matrice de Confusion\n{best_name}', fontsize=11)
ax1.set_ylabel('Réel')
ax1.set_xlabel('Prédit')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc = roc_auc_score(y_test, y_proba_best)
ax2.plot(fpr, tpr, color='#1976D2', linewidth=2.5, label=f'AUC = {auc:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax2.fill_between(fpr, tpr, alpha=0.1, color='#1976D2')
ax2.set_title('Courbe ROC', fontsize=11)
ax2.set_xlabel('Taux de Faux Positifs')
ax2.set_ylabel('Taux de Vrais Positifs')
ax2.legend(fontsize=11)

plt.suptitle(f'Évaluation — {best_name}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/best_model_eval.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Importance des variables (Random Forest)
if hasattr(best_model, 'feature_importances_'):
    imp_df = pd.DataFrame({
        'Feature':    X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance')

    plt.figure(figsize=(8, 5))
    bars = plt.barh(imp_df['Feature'], imp_df['Importance'],
                   color='#1976D2', edgecolor='white')
    plt.bar_label(bars, labels=[f'{v:.3f}' for v in imp_df['Importance']], padding=3)
    plt.title(f'Importance des Variables — {best_name}', fontsize=12)
    plt.xlabel('Importance (Gini)')
    plt.tight_layout()
    plt.savefig('../data/feature_importance.png', dpi=100, bbox_inches='tight')
    plt.show()

## 7. ⚙️ Amélioration — GridSearchCV (Hyperparameter Tuning)

In [ ]:
print('🔍 Recherche des meilleurs hyperparamètres (peut prendre 1-2 min)...')
param_grid = {
    'n_estimators':     [100, 200],
    'max_depth':        [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}
gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
gs.fit(X_train_sc, y_train)
print(f'✅ Meilleurs paramètres : {gs.best_params_}')
print(f'✅ Meilleur AUC (CV)    : {gs.best_score_:.4f}')

In [ ]:
final_model = gs.best_estimator_
y_pred_final  = final_model.predict(X_test_sc)
y_proba_final = final_model.predict_proba(X_test_sc)[:, 1]

print('=== RÉSULTATS APRÈS TUNING ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Précision : {precision_score(y_test, y_pred_final):.4f}')
print(f'Rappel    : {recall_score(y_test, y_pred_final):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_final):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_proba_final):.4f}')

## 8. 💾 Sauvegarde du modèle final

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(final_model, '../models/model.pkl')
joblib.dump(scaler,      '../models/scaler.pkl')

with open('../models/features.json', 'w') as f:
    json.dump(list(X.columns), f)

print('✅ Modèle sauvegardé    : ../models/model.pkl')
print('✅ Scaler sauvegardé    : ../models/scaler.pkl')
print('✅ Features sauvegardées: ../models/features.json')

## 9. 🧪 Test sur un patient exemple

In [ ]:
sample = pd.DataFrame([{
    'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72,
    'SkinThickness': 35, 'Insulin': 0, 'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627, 'Age': 50
}])
sample_sc = scaler.transform(sample)
proba     = final_model.predict_proba(sample_sc)[0]
print(f'🔬 Patient test (1er enregistrement du dataset) :')
print(f'   Probabilité diabète : {proba[1]*100:.1f}%')
print(f'   Probabilité non DT  : {proba[0]*100:.1f}%')
print(f'   Prédiction : {"⚠️  RISQUE DÉTECTÉ" if proba[1] >= 0.5 else "✅ Pas de risque significatif"}')
print(f'\n   (Valeur réelle dans le dataset : Outcome = 1 — diabétique)')

## ✅ Récapitulatif

| Bloc 2 | Compétence | ✓ |
|---|---|---|
| 1 | Définir le problème ML | Classification binaire (diabète Oui/Non) |
| 2 | Préparer les données | Imputation médiane, StandardScaler, train/test stratifié |
| 3 | Choisir et entraîner | 3 modèles : Logistic Regression, Random Forest, SVM |
| 4 | Évaluer | Accuracy, Précision, Rappel, F1, ROC-AUC, Matrice de confusion |
| 5 | Améliorer | GridSearchCV (5-fold CV), export model.pkl + scaler.pkl |